## 1. Setup and constants

In [110]:
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from scipy.stats import rankdata, spearmanr
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)

In [113]:
# ── Paths ─────────────────────────────────────────────
DATA_DIR = Path("../data/raw")
SUBMISSION = Path("../submission")

# ── General ───────────────────────────────────────────
RANDOM_STATE = 42
N_SPLIT = 5
TARGET = 'satisfaction'
ID_COLS = 'id'
TEST_SIZE = 0.2
# ── Columns, declared explicitly so nothing unexpected can slip into the model ─
NUM_COLS = [
    "Age", "Flight Distance",
    "Departure Delay in Minutes", "Arrival Delay in Minutes",
]

RATING_COLS = [
    "Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking",
    "Gate location", "Food and drink", "Online boarding", "Seat comfort",
    "Inflight entertainment", "On-board service", "Leg room service",
    "Baggage handling", "Checkin service", "Cleanliness",
]

CAT_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]
FEATURED_RAW = NUM_COLS + RATING_COLS + CAT_COLS
ENGINEERED = ["customer_rating","delay_diff","airplane_comfort"]

## 2. Load the data and check it

Three files: the competition `train` and `test`, and `orig`

In [87]:
def load_data():
    train = pd.read_csv(DATA_DIR/"train.csv")
    test = pd.read_csv(DATA_DIR/"test.csv")
    orig = pd.read_csv(DATA_DIR/"orig.csv")
    return train,test,orig
train,test,orig = load_data()

In [88]:
EXPECTED_COLS = {
    "train":[ID_COLS] + FEATURED_RAW + [TARGET],
    "test": [ID_COLS] + FEATURED_RAW,
    "orig": FEATURED_RAW + [TARGET]
}

for name,df in [("train",train),("test",test),("orig",orig)]:
    missing = set(EXPECTED_COLS[name]) - set(df.columns)
    unexpected = set(df.columns) - set(EXPECTED_COLS[name])
    assert not missing and not unexpected,f"{name}:missing  {missing} , unexpected {unexpected}"

train[TARGET] = train[TARGET].astype(int)
orig[TARGET] = orig[TARGET].astype(int)

summary = [] 
for name,df in [("train",train),("test",test),("orig",orig)]:
    summary.append({
        "table":name,
        "row":len(df),
        "columns":df.shape[1],
        "missing_values":int(df.isna().sum().sum()),
        "duplicate" : int(df.duplicated(subset=FEATURED_RAW).sum()),
        "satisfication": f"{df[TARGET].mean():.3%}" if TARGET in df else "-"})
pd.DataFrame(summary)

,table,row,columns,missing_values,duplicate,satisfication
0,train,699635,23,292,0,44.357%
1,test,299844,22,130,0,-
2,orig,129880,22,393,0,43.446%


In [89]:
type(train)

pandas.DataFrame

In [90]:
def imputer():
    train["Arrival Delay in Minutes"] = train["Arrival Delay in Minutes"].fillna(0)
    test["Arrival Delay in Minutes"] = test["Arrival Delay in Minutes"].fillna(0)
    orig["Arrival Delay in Minutes"] = orig["Arrival Delay in Minutes"].fillna(0)
    return train,test,orig
train,test,orig = imputer()

train[TARGET] = train[TARGET].astype(int)
orig[TARGET] = orig[TARGET].astype(int)

for col in CAT_COLS:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')
    orig[col] = orig[col].astype('category')

In [91]:
def add_features(df):
    df = df.copy()
    df["customer_rating"] = df[RATING_COLS].sum(axis=1)  
    df["delay_diff"]=df["Arrival Delay in Minutes"] + df["Departure Delay in Minutes"]
    df["airplane_comfort"] = df["Inflight wifi service"] + df["Food and drink"] + df["Seat comfort"] + df["Inflight entertainment"] + df["Leg room service"]
    return df
train = add_features(train)
test = add_features(test)
orig = add_features(orig)

In [95]:
print(x.dtypes[x.dtypes == "object"])


Series([], dtype: object)


In [134]:
# 1. any text columns left?
print(x.dtypes[x.dtypes == "object"])

# 2. force the 4 text columns to category, then rebuild x
for col in CAT_COLS:
    train[col] = train[col].astype("category")
    test[col]  = test[col].astype("category")

both = pd.concat([train, orig], ignore_index=True)
x = both.drop(columns=["id", "satisfaction"])
y = both["satisfaction"].astype(int)
x_test = test.drop(columns=["id"])

print(x.dtypes.value_counts())


Series([], dtype: object)
int64       18
float64      2
category     1
category     1
category     1
category     1
Name: count, dtype: int64


In [136]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def engine(x, y):
    scores = []
    for tr_idx, val_idx in skf.split(x, y):
        model = LGBMClassifier(verbose=-1)
        model.fit(x.iloc[tr_idx], y.iloc[tr_idx])
        p = model.predict_proba(x.iloc[val_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[val_idx], p))
    return scores

scores = engine(x, y)
print([round(s, 5) for s in scores], np.mean(scores).round(5))

[0.96393, 0.96284, 0.96333, 0.96313, 0.96442] 0.96353


In [137]:
x_raw  = x.drop(columns=NEW)
xt_raw = x_test[x_raw.columns].copy()
for col in CAT_COLS:
    xt_raw[col] = xt_raw[col].astype(x_raw[col].dtype)   # same categories as train

final = LGBMClassifier(**LGBM_PARAMS).fit(x_raw, y)
pred  = final.predict_proba(xt_raw)[:, 1]

sub = pd.DataFrame({"id": test["id"], "satisfaction": pred})
sub.to_csv("../submissions/submission.csv", index=False)
print(sub.shape, sub.head())

(299844, 2)        id  satisfaction
0  699635      0.269738
1  699636      0.813685
2  699637      0.915948
3  699638      0.080968
4  699639      0.049209
